In [1]:
import os
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import datetime as dt
from data_processor import DataReader, DataPrep
from scipy import stats

In [2]:
nyse = mcal.get_calendar('NYSE')
# Get holidays
holidays = nyse.holidays().holidays

### EDA

In [3]:
daily_data_path = r'data/daily_data'
intraday_data_path = r'data/intraday_data'

In [4]:
reader = DataReader()
intraday_df = reader.read_intraday_data(intraday_data_path)
daily_df = reader.read_daily_data(daily_data_path)
intraday_df.dropna(subset= 'CumReturnResid', inplace=True)

In [5]:
def query_id(df):
    
    df = df[df.CumReturnResid.isna()]
    df.drop_duplicates('Date', inplace= True)
    df['Diff'] = df.Date.diff().dt.days
    return df[['Date', 'Diff']].sort_values('Diff')

In [6]:
test_df = intraday_df.groupby('Id').apply(query_id)

In [7]:
test_df = intraday_df.query('Id == "BBG000BLM0V1"')#.drop_duplicates('Date').reset_index(drop= True).iloc[680:690]#query('Date == @query_date').copy()
test_df[test_df.CumReturnResid.isna()].drop_duplicates('Date').reset_index(drop= True).iloc[60:66]#query('Date == @query_date').copy()

,Date,Time,Id,CumReturnResid,CumReturnRaw,CumVolume


In [8]:
data_prep = DataPrep(intraday_df, daily_df)

In [9]:
target_df = data_prep.get_target()
target_df.dropna(subset='y', inplace = True)

### ID with different names

In [10]:
daily_df.query('Id == "BBG000CXD6X9"').SYMBOL.unique()

array(['HANS', 'MNST'], dtype=object)

In [11]:
MAD_by_Date = target_df.groupby('Date').apply(lambda x: stats.median_abs_deviation(x['y'])).to_frame()
MAD_by_Date.reset_index(inplace=True)
MAD_by_Date.columns = ['Date', 'MAD']

In [14]:
target_df = target_df.merge(MAD_by_Date,on= 'Date')

In [13]:
target_df

,Id,Date,y,EST_VOL
0,BBG000B9WH86,2010-01-04,-0.005550,0.17017
1,BBG000B9WH86,2010-01-05,-0.027280,0.18214
2,BBG000B9WH86,2010-01-06,-0.000220,0.18960
3,BBG000B9WH86,2010-01-07,0.002228,0.19430
4,BBG000B9WH86,2010-01-08,0.004919,0.20632
...,...,...,...,...
626425,GEN_EQ0140919900001000,2011-01-11,0.001833,0.10080
626426,GEN_EQ0140919900001000,2011-01-12,0.007235,0.07674
626427,GEN_EQ0140919900001000,2011-01-13,0.003934,0.07198
626428,GEN_EQ0140919900001000,2011-01-14,0.004534,0.07388


In [ ]:
import pandas as pd

# Create a pandas Series
s = pd.Series([1, 2, 3, 4, 5])

# Calculate the Mean Absolute Deviation (MAD)
mad = s.mad()

In [ ]:
daily_df.FREE_FLOAT_PERCENTAGE.hist()

In [ ]:
np.log(daily_df.MDV_63).hist()

In [ ]:
intraday_df.CumReturnResid

In [ ]:
intraday_df.query('Id == "BBG000MQ1SN9"')

### Features Prep